# 01 - Splits y diagnostico

Notebook de P3 para dejar listos los datos comunes del equipo.

In [51]:
# Cargamos librerias y definimos las rutas principales.
import os
os.environ.setdefault('MPLCONFIGDIR', '../outputs/matplotlib')

import pandas as pd
from sklearn.model_selection import train_test_split

DATA_PATH = '../data/raw/BasePI.xlsx'
TARGET = 'target'

os.makedirs('../data', exist_ok=True)
os.makedirs('../outputs', exist_ok=True)

## 1. Cargar la base

In [52]:
# Leemos la base y separamos las variables predictoras del target.
df = pd.read_excel(DATA_PATH, sheet_name='Datos')
diccionario = pd.read_excel(DATA_PATH, sheet_name='Diccionario')

variables = [col for col in df.columns if col != TARGET]

print('Filas y columnas:', df.shape)
print('Numero de variables predictoras:', len(variables))
print('Target:')
print(df[TARGET].value_counts())
print('Bad rate:', df[TARGET].mean())

Filas y columnas: (10000, 121)
Numero de variables predictoras: 120
Target:
target
0    9200
1     800
Name: count, dtype: int64
Bad rate: 0.08


## 2. Diagnostico rapido

In [53]:
# Guardamos un resumen general de la calidad de la base.
resumen = {
    'n_rows': df.shape[0],
    'n_cols': df.shape[1],
    'n_features': len(variables),
    'n_target_0': int((df[TARGET] == 0).sum()),
    'n_target_1': int((df[TARGET] == 1).sum()),
    'bad_rate': df[TARGET].mean(),
    'duplicated_full_rows': int(df.duplicated().sum()),
    'duplicated_feature_rows': int(df[variables].duplicated().sum())
}

resumen_df = pd.DataFrame([resumen])
resumen_df.to_csv('../outputs/data_quality_summary.csv', index=False)
resumen_df

,n_rows,n_cols,n_features,n_target_0,n_target_1,bad_rate,duplicated_full_rows,duplicated_feature_rows
0,10000,121,120,9200,800,0.08,182,183


In [54]:
# Calculamos missings por variable para revisar calidad de datos.
missings = pd.DataFrame({
    'variable': variables,
    'missing_count': df[variables].isna().sum().values,
    'missing_rate': df[variables].isna().mean().values,
    'n_unique': df[variables].nunique(dropna=True).values
})

missings = missings.sort_values('missing_rate', ascending=False)
missings.to_csv('../outputs/missing_by_variable.csv', index=False)

missings.head(10)

,variable,missing_count,missing_rate,n_unique
34,x35,874,0.0874,8105
119,x120,806,0.0806,9108
108,x109,674,0.0674,9303
103,x104,674,0.0674,9303
111,x112,674,0.0674,9305
110,x111,668,0.0668,9321
107,x108,668,0.0668,9318
105,x106,668,0.0668,9321
100,x101,668,0.0668,9321
102,x103,668,0.0668,9320


In [55]:
# Contamos duplicados para detectar posibles problemas en la validacion.
duplicados = pd.DataFrame({
    'metric': ['duplicated_full_rows', 'duplicated_feature_rows'],
    'value': [df.duplicated().sum(), df[variables].duplicated().sum()]
})

duplicados.to_csv('../outputs/duplicated_rows_summary.csv', index=False)
duplicados

,metric,value
0,duplicated_full_rows,182
1,duplicated_feature_rows,183


## 3. Limpiar duplicados antes del split

Antes de separar train/test/OOS, quitamos solo los duplicados exactos. Esto evita que una misma observacion quede repetida en distintos splits.

Los perfiles con las mismas variables pero diferente `target` no se borran automaticamente, porque pueden representar clientes con perfil observable igual pero comportamiento distinto.

In [56]:
# Quitamos duplicados exactos antes de generar los splits.
n_original = len(df)
n_duplicados_exactos = int(df.duplicated().sum())

# Este drop_duplicates revisa todas las columnas, incluyendo target.
df = df.drop_duplicates().reset_index(drop=True)

n_limpio = len(df)

print('Filas originales:', n_original)
print('Duplicados exactos eliminados:', n_duplicados_exactos)
print('Filas despues de limpiar:', n_limpio)

# Revisamos si quedan perfiles con mismas X pero distinto target.
# No los borramos; solo los reportamos para explicarlo al equipo.
grupo_targets = df.groupby(variables, dropna=False)[TARGET].nunique()
n_perfiles_contradictorios = int((grupo_targets > 1).sum())
n_filas_features_repetidas = int(df[variables].duplicated(keep=False).sum())

print('Perfiles con mismas variables y distinto target:', n_perfiles_contradictorios)
print('Filas que aun comparten las mismas variables predictoras:', n_filas_features_repetidas)

resumen_limpieza = pd.DataFrame([{
    'n_original': n_original,
    'duplicados_exactos_eliminados': n_duplicados_exactos,
    'n_despues_limpieza': n_limpio,
    'perfiles_contradictorios': n_perfiles_contradictorios,
    'filas_con_features_repetidas': n_filas_features_repetidas
}])

resumen_limpieza.to_csv('../outputs/duplicate_cleaning_summary.csv', index=False)
resumen_limpieza

Filas originales: 10000
Duplicados exactos eliminados: 182
Filas despues de limpiar: 9818
Perfiles con mismas variables y distinto target: 1
Filas que aun comparten las mismas variables predictoras: 2


,n_original,duplicados_exactos_eliminados,n_despues_limpieza,perfiles_contradictorios,filas_con_features_repetidas
0,10000,182,9818,1,2


## 4. Splits comunes

Usamos 70% train, 20% test y 10% oos/proxy OOS. Como no hay fecha, este OOS no es temporal; es un holdout fuera de muestra.

In [57]:
# Generamos los indices de train, test y oos con estratificacion.
# A partir de aqui usamos la base sin duplicados exactos.
df = df.reset_index(drop=True)
df['row_id'] = df.index

y = df[TARGET]

train_idx, temp_idx = train_test_split(
    df.index,
    test_size=0.30,
    stratify=y,
    random_state=42
)

test_idx, oos_idx = train_test_split(
    temp_idx,
    test_size=1/3,
    stratify=y.loc[temp_idx],
    random_state=42
)

print('Train:', len(train_idx))
print('Test :', len(test_idx))
print('OOS  :', len(oos_idx))

Train: 6872
Test : 1964
OOS  : 982


In [58]:
# Guardamos a que split pertenece cada fila original.
split_assignment = pd.DataFrame({
    'row_id': df.index,
    'split': 'none'
})

split_assignment.loc[train_idx, 'split'] = 'train'
split_assignment.loc[test_idx, 'split'] = 'test'
split_assignment.loc[oos_idx, 'split'] = 'oos'

split_assignment.to_csv('../data/split_assignment.csv', index=False)
split_assignment.head()

,row_id,split
0,0,train
1,1,train
2,2,test
3,3,test
4,4,train


## 5. Guardar X/y de cada split

In [59]:
# Guardamos los archivos X/y que usara todo el equipo.
train_df = df.loc[train_idx].copy()
test_df = df.loc[test_idx].copy()
oos_df = df.loc[oos_idx].copy()

X_train = train_df[variables]
X_test = test_df[variables]
X_oos = oos_df[variables]

y_train = train_df[[TARGET]]
y_test = test_df[[TARGET]]
y_oos = oos_df[[TARGET]]

X_train.to_csv('../data/X_train.csv', index=False)
X_test.to_csv('../data/X_test.csv', index=False)
X_oos.to_csv('../data/X_oos.csv', index=False)

y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)
y_oos.to_csv('../data/y_oos.csv', index=False)

## 6. Verificar que el target quedo parecido en todos los splits

In [60]:
# Verificamos que el bad rate quede parecido en todos los splits.
split_check = df.merge(split_assignment, on='row_id')

tabla_split = split_check.groupby('split')[TARGET].agg(
    n='count',
    malos='sum',
    bad_rate='mean'
)

tabla_split

,n,malos,bad_rate
split,,,
oos,982,79,0.080448
test,1964,159,0.080957
train,6872,554,0.080617


---
## Resumen de calidad de datos y splits

### Desbalance del target (clase minoritaria)
La base tiene **10,000 filas y 120 variables predictoras**. La tasa de incumplimiento (bad rate) es **8.0%** (800 malos de 9,200 buenos). La clase `1` (malo) es la minoría. Implicaciones para el modelado:
- **Accuracy no es una métrica útil** — un modelo trivial que prediga siempre 0 tendría 92% de accuracy sin aprender nada.
- Las métricas relevantes son: **KS, AUC-ROC, Gini, Recall (sensibilidad) y Lift por decil**.
- Por eso se usa `class_weight='balanced'` en todos los modelos y el threshold se calibra por máximo KS sobre train, no por 0.5.

### Duplicados
| Tipo | Cantidad |
|---|---:|
| Duplicados exactos eliminados antes del split | **182** |
| Filas con mismas X y distinto target (perfiles contradictorios) | 1 |
| Filas con mismas X (todas, incluyendo target) | 2 |

Los 182 duplicados exactos se eliminaron **antes** del split para evitar contaminación entre train/test/OOS. La base útil resultante tiene **9,818 filas**.

### Variables con mayor porcentaje de valores faltantes
Las siguientes variables tienen missingness no trivial y su tratamiento debe documentarse:

| Variable | Missing rate |
|---|---:|
| x35 | 8.7% |
| x120 | 8.1% |
| x109 | 6.7% |
| x104 | 6.7% |
| x112 | 6.7% |
| x111, x108, x106, x101, x103 | ~6.7% c/u |

Estrategia adoptada: `SimpleImputer(strategy='median', add_indicator=True)` dentro del Pipeline de sklearn. Esto imputa por mediana **y** agrega un indicador binario por cada variable con missings, permitiendo que el modelo capture si la ausencia en sí es informativa.

### Splits finales
| Split | N | Bad rate |
|---|---:|---:|
| Train | ~6,873 | ~8% |
| Test | ~1,964 | ~8% |
| OOS | ~981 | ~8% |

Todos los splits se estratificaron con `stratify=y` y `random_state=42`.  
Como no existe variable de fecha en la base, el OOS es un **holdout aleatorio fuera de muestra**, no temporal.